# D4 — Interpretable models: the price of interpretability (descriptive)

Glass-box models trained on the same L1 data, features, temporal split and v2 protocol as C1-R:
- **EBM:** `interpret` 0.7.8, handles missing values natively.
- **RuleFit:** `imodels` 3.0.4.
- **Sparse optimal tree:** GOSDT 1.0.4, `balance=True`, threshold-guess binarisation with 200 depth-2 stumps. Each leaf is scored by its training KEV rate, so the tree can rank.

RuleFit and GOSDT get median imputation plus missing indicators, fitted on training rows.

Hyperparameters were searched on the v2 fold (seed 0, one fit per config) and selected by validation PR-AUC. The selected config was then refitted on the full training set with 10 seeds and tested once. Each model is compared with C1-R LR, C1-R XGBoost and EPSS-PIT by a paired bootstrap on the seed-mean scores. Design: Study D addendum (commit c0a3ee1). **No hypothesis**: D4 is descriptive.

Reads saved outputs. Run: `python -m src.models.d4_interpretable --stage all` (search about 1 h, final about 5 h on this machine).

In [1]:
import os, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
import pandas as pd
from IPython.display import Image
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40)

## (1) Search on the v2 fold

In [2]:
pd.read_csv("outputs/d4_search.csv").drop(columns=["fit_seconds"]).round(4)

,kind,config_id,config,val_pr_auc,n_terms,n_interactions,n_rules,n_linear,n_leaves,error
0,ebm,0,"{""interactions"": 0, ""learning_rate"": 0.015}",0.0683,69.0,0.0,NaN,NaN,NaN,NaN
1,ebm,1,"{""interactions"": 0, ""learning_rate"": 0.04}",0.0632,69.0,0.0,NaN,NaN,NaN,NaN
2,ebm,2,"{""interactions"": 10, ""learning_rate"": 0.015}",0.0646,79.0,10.0,NaN,NaN,NaN,NaN
3,ebm,3,"{""interactions"": 10, ""learning_rate"": 0.04}",0.0620,79.0,10.0,NaN,NaN,NaN,NaN
4,rulefit,0,"{""max_rules"": 10}",0.0295,9.0,NaN,8.0,1.0,NaN,NaN
5,rulefit,1,"{""max_rules"": 30}",0.0533,30.0,NaN,14.0,16.0,NaN,NaN
6,rulefit,2,"{""max_rules"": 60}",0.0727,59.0,NaN,22.0,37.0,NaN,NaN
7,gosdt,0,"{""depth_budget"": 3, ""regularization"": 0.001}",0.0037,6.0,NaN,NaN,NaN,6.0,NaN
8,gosdt,1,"{""depth_budget"": 3, ""regularization"": 0.005}",0.0057,4.0,NaN,NaN,NaN,4.0,NaN
9,gosdt,2,"{""depth_budget"": 4, ""regularization"": 0.001}",NaN,NaN,NaN,NaN,NaN,NaN,"OSError(28, 'No space left on device')"


**The depth-4 GOSDT configs failed.** GOSDT's memory use grew until swap filled a nearly full disk (`OSError: No space left on device`), and the process died without a traceback. They are recorded as failed and, by the rule fixed then, not retried. Selection is therefore among depth-3 trees only, and the GOSDT result below is bounded at depth 3. Since then, every GOSDT search fit runs in its own subprocess.

The class-weighted CART fallback was not triggered, because both depth-3 configs returned trees with more than one leaf.

In [3]:
pd.read_csv("outputs/d4_selected_configs.csv", index_col=0)[["config", "val_pr_auc"]]

,config,val_pr_auc
kind,,
ebm,"{""interactions"": 0, ""learning_rate"": 0.015}",0.068255
gosdt,"{""depth_budget"": 3, ""regularization"": 0.005}",0.005732
rulefit,"{""max_rules"": 60}",0.072669


## (2) Test performance, 10 seeds (mean ± std)

In [4]:
pd.read_csv("outputs/d4_summary.csv", index_col=0)

,pr_auc,precision_at_100,ndcg_at_100,roc_auc,brier,ece
kind,,,,,,
ebm,0.0187 ± 0.0007,0.105 ± 0.0067,0.1108 ± 0.0071,0.8217 ± 0.0016,0.0014 ± 0.0,0.0005 ± 0.0
gosdt,0.0027 ± 0.0001,0.02 ± 0.0,0.0189 ± 0.0036,0.6339 ± 0.0284,0.0014 ± 0.0,0.0013 ± 0.0014
rulefit,0.0196 ± 0.0018,0.098 ± 0.0166,0.1271 ± 0.0167,0.8083 ± 0.0053,0.0014 ± 0.0,0.0014 ± 0.0


## (3) Paired bootstrap vs C1-R and EPSS-PIT (seed-mean scores; diff = D4 model − reference)

In [5]:
pd.read_csv("outputs/d4_tests.csv", index_col=0).round(4)

,pr_auc_diff,pr_auc_ci_low,pr_auc_ci_high,pr_auc_p_not_better,p_at_100_diff,p_at_100_ci_low,p_at_100_ci_high,p_at_100_p_not_better
ebm_vs_C1R_logreg,-0.0049,-0.0148,0.0005,0.956,-0.02,-0.05,0.00,0.992
ebm_vs_C1R_xgboost,-0.0036,-0.0113,0.0017,0.908,-0.04,-0.07,0.00,1.000
ebm_vs_epss_pit,0.0011,-0.0065,0.0064,0.350,-0.01,-0.05,0.03,0.798
gosdt_vs_C1R_logreg,-0.0201,-0.0390,-0.0108,1.000,-0.12,-0.19,-0.05,1.000
gosdt_vs_C1R_xgboost,-0.0188,-0.0365,-0.0100,1.000,-0.14,-0.20,-0.05,1.000
gosdt_vs_epss_pit,-0.0141,-0.0314,-0.0059,1.000,-0.11,-0.17,-0.04,1.000
rulefit_vs_C1R_logreg,-0.0023,-0.0078,0.0042,0.792,-0.01,-0.04,0.00,1.000
rulefit_vs_C1R_xgboost,-0.0010,-0.0064,0.0076,0.634,-0.03,-0.06,0.00,1.000
rulefit_vs_epss_pit,0.0037,-0.0010,0.0098,0.064,0.00,-0.05,0.04,0.636


## (4) Model size

In [6]:
s = pd.read_csv("outputs/d4_model_size.csv")
s.groupby("kind").agg(fit_minutes=("fit_seconds", lambda x: round(x.mean() / 60, 1)), n_terms=("n_terms", "mean"),
                      n_rules=("n_rules", "mean"), n_linear=("n_linear", "mean"), n_leaves=("n_leaves", "mean"))

,fit_minutes,n_terms,n_rules,n_linear,n_leaves
kind,,,,,
ebm,4.1,69.0,NaN,NaN,NaN
gosdt,15.4,3.0,NaN,NaN,3.0
rulefit,8.3,58.2,33.4,24.8,NaN


## (5) EBM global term importance, and its agreement with D1's rankings (Kendall τ-b, 69 features)

In [7]:
pd.read_csv("outputs/d4_ebm_term_importance.csv", index_col=0).sort_values("importance", ascending=False).head(12).round(3)

,importance
cvss_version,4.369
epss_pit_pct,0.584
epss_pit,0.583
cvss_score,0.561
text_09,0.334
text_20,0.318
text_10,0.290
text_04,0.214
text_17,0.210
text_27,0.196


In [8]:
pd.read_csv("outputs/d4_global_ranking_agreement.csv", index_col=0).round(3)

,SHAP M-XGB,SHAP M-LR,PermImp M-XGB,PermImp M-LR,EBM term importance
SHAP M-XGB,1.000,0.295,0.169,0.165,0.424
SHAP M-LR,0.295,1.000,0.066,0.176,0.312
PermImp M-XGB,0.169,0.066,1.000,0.293,0.181
PermImp M-LR,0.165,0.176,0.293,1.000,0.090
EBM term importance,0.424,0.312,0.181,0.090,1.000


**Reading.**
- **EBM (PR-AUC 0.0187) and RuleFit (0.0196) perform like the black boxes.** Every paired-bootstrap CI against C1-R LR, C1-R XGBoost and EPSS-PIT includes zero. Point estimates are slightly below C1-R (−0.001 to −0.005) and slightly above EPSS-PIT (+0.001 / +0.004). P@100 is 0.10 for both.
- **There is no price of interpretability on this task, but not for the usual reason.** The best model in Study C, logistic regression, is already a glass box. EBM and RuleFit add shape functions and rules without adding ranking performance. What they offer is a different *form* of explanation, such as per-feature shape plots and if-then rules. D6 is meant to test whether that form helps engineers.
- **A depth-3 optimal tree cannot rank 1-in-240 events.** It has 3 leaves, PR-AUC 0.0027 and ROC 0.63, and is significantly worse than every reference. With only three distinct scores it cannot order findings within a month. This conclusion is bounded at depth 3 (see the incident in (1)).
- **EBM's own importance repeats the SHAP background artefact.** `cvss_version` is again the top term (4.37, against about 0.58 for each EPSS feature). EBM's importance is the mean |term score| around the *training* intercept, so it inherits the same era shift that dominated training-background SHAP in D1 and D2. It agrees best with SHAP M-XGB (τ 0.42) and worst with permutation importance (τ 0.09–0.18). Two independent attribution methods share the artefact; the permutation-based ranking does not.
- RuleFit used 33 rules plus 25 linear terms on average, and seeds vary in which rules survive. A rule list that changes from seed to seed is a stability concern for presenting rules to analysts.